In [ ]:
import os
import gc
import json
import librosa
import numpy as np
from tqdm import tqdm
from pathlib import Path
import warnings
import torch
import psutil
from datetime import datetime
import zipfile
import shutil
warnings.filterwarnings("ignore")

# --- MEMORY MANAGEMENT ---
def clear_memory():
    """Clear memory and run garbage collection"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def get_memory_usage():
    """Get current memory usage"""
    process = psutil.Process(os.getpid())
    mem_gb = process.memory_info().rss / 1024 / 1024 / 1024
    if torch.cuda.is_available():
        gpu_mem = torch.cuda.memory_allocated() / 1024**3
        return f"RAM: {mem_gb:.2f}GB, GPU: {gpu_mem:.2f}GB"
    return f"RAM: {mem_gb:.2f}GB"

# --- PATH SETUP ---
SOURCE_DIR = "/kaggle/input/fakeav/AUDIO/AUDIO"
WORK_DIR = "/kaggle/working"
OUTPUT_BASE_DIR = os.path.join(WORK_DIR, "MFCC_Features")
PROGRESS_FILE = os.path.join(WORK_DIR, "audio_progress.json")

# Create output directories for multiclass classification
CLASS_DIRS = {
    'RealVideo-RealAudio': os.path.join(OUTPUT_BASE_DIR, "RealVideo-RealAudio"),
    'FakeVideo-FakeAudio': os.path.join(OUTPUT_BASE_DIR, "FakeVideo-FakeAudio"), 
    'RealVideo-FakeAudio': os.path.join(OUTPUT_BASE_DIR, "RealVideo-FakeAudio"),
    'FakeVideo-RealAudio': os.path.join(OUTPUT_BASE_DIR, "FakeVideo-RealAudio")
}

# Create all output directories
for class_dir in CLASS_DIRS.values():
    os.makedirs(class_dir, exist_ok=True)

# --- CONFIG ---
BATCH_SIZE = 500       # Increased batch size as requested
CLEAR_EVERY = 50       # Clear memory every N files
MFCC_FEATURES = 40
AUDIO_DURATION = 10    # Limit audio duration
SR = 16000            # Sample rate

# --- PROGRESS TRACKING ---
def load_progress():
    """Load processing progress from file"""
    if os.path.exists(PROGRESS_FILE):
        try:
            with open(PROGRESS_FILE, 'r') as f:
                return json.load(f)
        except:
            pass
    return {
        'processed_files': set(),
        'total_processed': 0,
        'failed_files': [],
        'class_counts': {
            'RealVideo-RealAudio': 0,
            'FakeVideo-FakeAudio': 0,
            'RealVideo-FakeAudio': 0,
            'FakeVideo-RealAudio': 0
        },
        'start_time': datetime.now().isoformat()
    }

def save_progress(progress):
    """Save processing progress to file"""
    progress_copy = progress.copy()
    progress_copy['processed_files'] = list(progress['processed_files'])
    progress_copy['last_update'] = datetime.now().isoformat()
    
    with open(PROGRESS_FILE, 'w') as f:
        json.dump(progress_copy, f, indent=2)

# --- GET FILES ---
all_wav_files = sorted(list(Path(SOURCE_DIR).rglob("*.wav")))
total_files = len(all_wav_files)
print(f"🎯 Total .wav files found: {total_files}")

# Load existing progress
progress = load_progress()
processed_files = set(progress['processed_files'])
print(f"📋 Previously processed: {len(processed_files)} files")
print(f"🔄 Remaining to process: {total_files - len(processed_files)} files")

# --- MULTICLASS LABEL DETECTION ---
def get_multiclass_label(file_path):
    """Determine the multiclass label based on filename and path"""
    filename = file_path.name.lower()
    parent_path = str(file_path.parent).lower()
    full_path = str(file_path).lower()
    
    # Initialize flags
    video_fake = False
    audio_fake = False
    
    # Video authenticity indicators
    video_fake_indicators = ['fakevideo', 'fake_video', 'fv', 'deepfake', 'synthetic_video']
    video_real_indicators = ['realvideo', 'real_video', 'rv', 'original_video', 'authentic_video']
    
    # Audio authenticity indicators  
    audio_fake_indicators = ['fakeaudio', 'fake_audio', 'fa', 'synthetic_audio', 'cloned_audio']
    audio_real_indicators = ['realaudio', 'real_audio', 'ra', 'original_audio', 'authentic_audio']
    
    # Check for explicit video indicators
    for indicator in video_fake_indicators:
        if indicator in filename or indicator in parent_path:
            video_fake = True
            break
    
    if not video_fake:
        for indicator in video_real_indicators:
            if indicator in filename or indicator in parent_path:
                video_fake = False
                break
    
    # Check for explicit audio indicators
    for indicator in audio_fake_indicators:
        if indicator in filename or indicator in parent_path:
            audio_fake = True
            break
    
    if not audio_fake:
        for indicator in audio_real_indicators:
            if indicator in filename or indicator in parent_path:
                audio_fake = False
                break
    
    # Pattern-based detection if no explicit indicators
    if 'rv-ra' in filename or 'realvideo_realaudio' in filename:
        return 'RealVideo-RealAudio'
    elif 'fv-fa' in filename or 'fakevideo_fakeaudio' in filename:
        return 'FakeVideo-FakeAudio'
    elif 'rv-fa' in filename or 'realvideo_fakeaudio' in filename:
        return 'RealVideo-FakeAudio'
    elif 'fv-ra' in filename or 'fakevideo_realaudio' in filename:
        return 'FakeVideo-RealAudio'
    
    # Directory-based detection
    if 'realvideo' in parent_path and 'realaudio' in parent_path:
        return 'RealVideo-RealAudio'
    elif 'fakevideo' in parent_path and 'fakeaudio' in parent_path:
        return 'FakeVideo-FakeAudio'
    elif 'realvideo' in parent_path and 'fakeaudio' in parent_path:
        return 'RealVideo-FakeAudio'
    elif 'fakevideo' in parent_path and 'realaudio' in parent_path:
        return 'FakeVideo-RealAudio'
    
    # Fallback: try to determine from general fake/real indicators
    general_fake_indicators = ['fake', 'deepfake', 'synthetic', 'generated', 'spoof']
    general_real_indicators = ['real', 'original', 'authentic', 'genuine']
    
    is_fake = False
    for indicator in general_fake_indicators:
        if indicator in filename or indicator in parent_path:
            is_fake = True
            break
    
    if is_fake:
        # If we detect it's fake but can't determine video/audio split, assume both fake
        return 'FakeVideo-FakeAudio'
    else:
        # Default to real-real if no clear indicators
        return 'RealVideo-RealAudio'

# --- AUDIO EXTRACTION FUNCTION ---
def extract_mfcc_features(wav_path, save_path):
    """Extract MFCC features from wave audio file"""
    if os.path.exists(save_path): 
        return True
        
    try:
        # Load audio with librosa directly from wave file
        y, sr = librosa.load(str(wav_path), sr=SR, duration=AUDIO_DURATION)
        if len(y) < 1024:  # Too short audio
            return False
            
        # Extract MFCC features
        mfcc = librosa.feature.mfcc(
            y=y, 
            sr=sr, 
            n_mfcc=MFCC_FEATURES,
            hop_length=512, 
            n_fft=2048,
            window='hann'
        )
        
        # Also extract additional features for better representation
        spectral_centroids = librosa.feature.spectral_centroid(y=y, sr=sr)
        zero_crossings = librosa.feature.zero_crossing_rate(y)
        
        # Combine features
        features = np.vstack([mfcc, spectral_centroids, zero_crossings])
        
        # Save as compressed numpy array
        np.save(save_path, features.astype(np.float32))
        return True
        
    except Exception as e:
        print(f"❌ MFCC extraction failed for {wav_path.name}: {str(e)}")
        return False

# --- ZIP CREATION FUNCTION ---
def create_zip_file():
    """Create a zip file of all extracted features"""
    print("\n📦 Creating ZIP file for download...")
    
    zip_path = os.path.join(WORK_DIR, "MFCC_Features.zip")
    
    try:
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
            # Add all files from the output directory
            for root, dirs, files in os.walk(OUTPUT_BASE_DIR):
                for file in files:
                    file_path = os.path.join(root, file)
                    # Calculate relative path for zip
                    arcname = os.path.relpath(file_path, OUTPUT_BASE_DIR)
                    zipf.write(file_path, arcname)
        
        # Get zip file size
        zip_size = os.path.getsize(zip_path) / (1024 * 1024)  # MB
        print(f"✅ ZIP file created: {zip_path}")
        print(f"📊 ZIP file size: {zip_size:.2f} MB")
        
        return zip_path
        
    except Exception as e:
        print(f"❌ Failed to create ZIP file: {str(e)}")
        return None

# --- MAIN PROCESSING FUNCTION ---
def process_audio_files():
    """Process all wave files for audio MFCC extraction"""
    
    remaining_files = [f for f in all_wav_files 
                      if f.stem not in processed_files]
    
    if not remaining_files:
        print("✅ All audio files already processed!")
        return 0, 0  # Return success and failed counts
    
    print(f"🚀 Starting MFCC extraction for {len(remaining_files)} files")
    print(f"💾 {get_memory_usage()}")
    
    successful = 0
    failed = 0
    class_counts = progress.get('class_counts', {
        'RealVideo-RealAudio': 0,
        'FakeVideo-FakeAudio': 0,
        'RealVideo-FakeAudio': 0,
        'FakeVideo-RealAudio': 0
    })
    
    try:
        for start in range(0, len(remaining_files), BATCH_SIZE):
            batch = remaining_files[start:start + BATCH_SIZE]
            batch_success = 0
            
            print(f"\n🔄 Processing batch {start//BATCH_SIZE + 1}/{(len(remaining_files)-1)//BATCH_SIZE + 1}")
            print(f"📁 Files {start + 1} to {start + len(batch)} of {len(remaining_files)}")
            
            for i, file_path in enumerate(tqdm(batch, desc="Extracting MFCC")):
                filename = file_path.stem
                
                # Skip if already processed
                if filename in processed_files:
                    continue
                
                # Determine multiclass label
                label = get_multiclass_label(file_path)
                output_dir = CLASS_DIRS[label]
                
                # Output path
                audio_out = os.path.join(output_dir, filename + ".npy")
                
                # Process audio
                try:
                    if extract_mfcc_features(file_path, audio_out):
                        batch_success += 1
                        successful += 1
                        class_counts[label] += 1
                    else:
                        failed += 1
                        progress['failed_files'].append(filename)
                    
                    # Mark as processed regardless of success
                    processed_files.add(filename)
                    
                except Exception as e:
                    print(f"❌ Error processing {filename}: {str(e)}")
                    failed += 1
                    progress['failed_files'].append(filename)
                    processed_files.add(filename)
                    continue
                
                # Memory management
                if (i + 1) % CLEAR_EVERY == 0:
                    clear_memory()
            
            # Update progress after each batch
            progress['processed_files'] = processed_files
            progress['total_processed'] = len(processed_files)
            progress['class_counts'] = class_counts
            save_progress(progress)
            clear_memory()
            
            print(f"✅ Batch completed: Success={batch_success}, Failed={len(batch)-batch_success}")
            print(f"📊 Overall progress: {len(processed_files)}/{total_files} files")
            print(f"📊 Class distribution:")
            for class_name, count in class_counts.items():
                print(f"   {class_name}: {count}")
            print(f"💾 {get_memory_usage()}")
    
    except KeyboardInterrupt:
        print("\n⚠️ Processing interrupted by user")
        progress['processed_files'] = processed_files
        progress['total_processed'] = len(processed_files)
        progress['class_counts'] = class_counts
        save_progress(progress)
        print("💾 Progress saved. You can resume later.")
        return successful, failed  # Return counts even when interrupted
    except Exception as e:
        print(f"\n❌ Unexpected error: {str(e)}")
        progress['processed_files'] = processed_files
        progress['total_processed'] = len(processed_files)
        progress['class_counts'] = class_counts
        save_progress(progress)
        return successful, failed  # Return counts even on error
    
    return successful, failed

# --- RUN PROCESSING ---
if __name__ == "__main__":
    print("🎵 FakeAV Multiclass Audio MFCC Extraction")
    print("🏷️  Classes: RealVideo-RealAudio, FakeVideo-FakeAudio, RealVideo-FakeAudio, FakeVideo-RealAudio")
    print("=" * 80)
    
    # Handle the case where process_audio_files might return None
    result = process_audio_files()
    if result is not None:
        successful, failed = result
    else:
        # Fallback values if somehow None is returned
        successful, failed = 0, 0
    
    # Final summary
    print("\n🎉 Audio processing completed!")
    print(f"✅ Successfully processed: {successful}")
    print(f"❌ Failed to process: {failed}")
    print(f"📁 Total files processed: {len(processed_files)}")
    print(f"💾 Final memory usage: {get_memory_usage()}")
    
    # Show detailed statistics about extracted files
    print(f"\n📊 Detailed Class Distribution:")
    total_features = 0
    for class_name, class_dir in CLASS_DIRS.items():
        class_files = list(Path(class_dir).glob("*.npy"))
        count = len(class_files)
        total_features += count
        print(f"   {class_name}: {count} files")
    
    print(f"\n🎵 Total MFCC files saved: {total_features}")
    
    # Calculate storage statistics
    if total_features > 0:
        all_feature_files = []
        for class_dir in CLASS_DIRS.values():
            all_feature_files.extend(list(Path(class_dir).glob("*.npy")))
        
        if all_feature_files:
            sample_size = os.path.getsize(all_feature_files[0]) / 1024  # KB
            total_size = sum(os.path.getsize(f) for f in all_feature_files) / (1024*1024)  # MB
            print(f"📊 Average file size: {sample_size:.2f} KB")
            print(f"📊 Total data size: {total_size:.2f} MB")
    
    # Create ZIP file for download
    zip_path = create_zip_file()
    if zip_path:
        print(f"\n📦 Ready for download: {zip_path}")
    
    # Clean up progress file if everything is done
    if len(processed_files) >= total_files:
        try:
            os.remove(PROGRESS_FILE)
            print("🧹 Cleaned up progress file")
        except:
            pass
    
    print("\n🎯 Processing Summary:")
    print(f"{'='*50}")
    print(f"Total files found: {total_files}")
    print(f"Successfully processed: {successful}")
    print(f"Failed: {failed}")
    print(f"Features extracted: {total_features}")
    if zip_path:
        print(f"Download ready: MFCC_Features.zip")
    print(f"{'='*50}")

🎯 Total .wav files found: 21544
📋 Previously processed: 0 files
🔄 Remaining to process: 21544 files
🎵 FakeAV Multiclass Audio MFCC Extraction
🏷️  Classes: RealVideo-RealAudio, FakeVideo-FakeAudio, RealVideo-FakeAudio, FakeVideo-RealAudio
🚀 Starting MFCC extraction for 21544 files
💾 RAM: 0.52GB

🔄 Processing batch 1/44
📁 Files 1 to 500 of 21544


Extracting MFCC: 100%|██████████| 500/500 [00:43<00:00, 11.41it/s]


✅ Batch completed: Success=487, Failed=13
📊 Overall progress: 487/21544 files
📊 Class distribution:
   RealVideo-RealAudio: 0
   FakeVideo-FakeAudio: 487
   RealVideo-FakeAudio: 0
   FakeVideo-RealAudio: 0
💾 RAM: 0.74GB

🔄 Processing batch 2/44
📁 Files 501 to 1000 of 21544


Extracting MFCC: 100%|██████████| 500/500 [00:23<00:00, 20.89it/s]


✅ Batch completed: Success=472, Failed=28
📊 Overall progress: 959/21544 files
📊 Class distribution:
   RealVideo-RealAudio: 0
   FakeVideo-FakeAudio: 959
   RealVideo-FakeAudio: 0
   FakeVideo-RealAudio: 0
💾 RAM: 0.74GB

🔄 Processing batch 3/44
📁 Files 1001 to 1500 of 21544


...

Extracting MFCC: 100%|██████████| 500/500 [00:28<00:00, 17.67it/s]


✅ Batch completed: Success=486, Failed=14
📊 Overall progress: 2417/21544 files
📊 Class distribution:
   RealVideo-RealAudio: 0
   FakeVideo-FakeAudio: 2417
   RealVideo-FakeAudio: 0
   FakeVideo-RealAudio: 0
💾 RAM: 0.74GB

🔄 Processing batch 6/44
📁 Files 2501 to 3000 of 21544


Extracting MFCC: 100%|██████████| 500/500 [00:24<00:00, 20.20it/s]


✅ Batch completed: Success=457, Failed=43
📊 Overall progress: 2874/21544 files
📊 Class distribution:
   RealVideo-RealAudio: 0
   FakeVideo-FakeAudio: 2874
   RealVideo-FakeAudio: 0
   FakeVideo-RealAudio: 0
💾 RAM: 0.74GB

🔄 Processing batch 7/44
📁 Files 3001 to 3500 of 21544


Extracting MFCC:  93%|█████████▎| 467/500 [00:23<00:01, 21.09it/s]